# Programación funcional · Avanzado

Composición de `map` y `filter`, estadística, rendimiento y corrutinas.

## 1. Buscador de inmuebles

Dada una lista de inmuebles, escribir una función que reciba la lista y un presupuesto y devuelva los inmuebles cuyo precio no lo supere, añadiendo a cada uno su precio. Precio base: `(metros·1000 + habitaciones·5000 + garaje·15000) · (1 − antigüedad/100)`; en la zona B se multiplica por 1.5.

In [1]:
from datetime import date

inmuebles = [
    {"año": 2000, "metros": 100, "habitaciones": 3, "garaje": True, "zona": "A"},
    {"año": 2012, "metros": 60, "habitaciones": 2, "garaje": True, "zona": "B"},
    {"año": 1980, "metros": 120, "habitaciones": 4, "garaje": False, "zona": "A"},
    {"año": 2005, "metros": 75, "habitaciones": 3, "garaje": True, "zona": "B"},
    {"año": 2015, "metros": 90, "habitaciones": 2, "garaje": False, "zona": "A"},
]

def con_precio(inmueble):
    antiguedad = date.today().year - inmueble["año"]
    precio = (inmueble["metros"] * 1000 + inmueble["habitaciones"] * 5000 + inmueble["garaje"] * 15000) * (1 - antiguedad / 100)
    return {**inmueble, "precio": precio * (1.5 if inmueble["zona"] == "B" else 1)}

def busca(inmuebles, presupuesto):
    return list(filter(lambda i: i["precio"] <= presupuesto, map(con_precio, inmuebles)))

for inmueble in busca(inmuebles, 100000):
    print(inmueble)

{'año': 2000, 'metros': 100, 'habitaciones': 3, 'garaje': True, 'zona': 'A', 'precio': 96200.0}
{'año': 1980, 'metros': 120, 'habitaciones': 4, 'garaje': False, 'zona': 'A', 'precio': 75600.0}
{'año': 2015, 'metros': 90, 'habitaciones': 2, 'garaje': False, 'zona': 'A', 'precio': 89000.0}


## 2. Valores atípicos

Escribir una función que reciba una muestra y devuelva los valores atípicos: aquellos cuya puntuación típica $\frac{x-\bar{x}}{\sigma}$ es mayor que 3 o menor que −3.

In [2]:
def atipicos(muestra):
    media = sum(muestra) / len(muestra)
    sigma = (sum((x - media) ** 2 for x in muestra) / len(muestra)) ** 0.5
    return [x for x in muestra if abs((x - media) / sigma) > 3]

print(atipicos([1, 2, 3, 2, 1, 3, 2, 2, 1, 3, 2, 2, 3, 1, 2, 50]))

[50]


## 3. Benchmark con `timeit`

Encontrar dos formas de hacer la misma tarea, comprobar con `timeit` cuál es más rápida y medir también el consumo de memoria con `tracemalloc`.

In [3]:
import timeit
import tracemalloc

def con_bucle(n):
    resultado = []
    for i in range(n):
        resultado.append(i * i)
    return resultado

def con_comprension(n):
    return [i * i for i in range(n)]

def con_generador(n):
    return sum(i * i for i in range(n))

for f in (con_bucle, con_comprension):
    print(f"{f.__name__:16} {timeit.timeit(lambda: f(100_000), number=50):.3f} s")

for f in (con_comprension, con_generador):
    tracemalloc.start()
    f(100_000)
    pico = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    print(f"{f.__name__:16} pico de memoria {pico / 1024:.0f} KiB")

con_bucle        0.150 s
con_comprension  0.138 s
con_comprension  pico de memoria 3907 KiB
con_generador    pico de memoria 1 KiB


## 4. Peticiones asíncronas con límite

Simular peticiones a una API con `asyncio`, procesando como máximo `MAX_TASKS` a la vez y gestionando las excepciones de cada petición para que un fallo no arruine el resto.

In [4]:
import asyncio
import random

MAX_TASKS = 3

async def fetch(usuario):
    # simula latencia y fallos de red
    await asyncio.sleep(random.uniform(0.05, 0.2))
    if random.random() < 0.2:
        raise ConnectionError(f"fallo al pedir {usuario}")
    return random.randint(0, 1000)

async def fetch_all(usuarios):
    semaforo = asyncio.Semaphore(MAX_TASKS)

    async def limitado(usuario):
        async with semaforo:
            return await fetch(usuario)

    resultados = await asyncio.gather(*(limitado(u) for u in usuarios), return_exceptions=True)
    return dict(zip(usuarios, resultados))

# Jupyter permite await directamente en la celda
resultados = await fetch_all([f"user{i}" for i in range(8)])
for usuario, seguidores in resultados.items():
    print(usuario, seguidores)

user0 282
user1 499
user2 120
user3 fallo al pedir user3
user4 526
user5 fallo al pedir user5
user6 753
user7 115


## 5. `map`, `filter` y `reduce`

Con `countries = ['Estonia', 'Finland', 'Sweden', 'Denmark', 'Norway', 'Iceland']`, `names = ['Asabeneh', 'Lidiya', 'Ermias', 'Abraham']` y los números del 1 al 10: pasar países y nombres a mayúsculas y números al cuadrado con `map`; filtrar países que contienen `land`, que tienen exactamente 6 caracteres, 6 o más, o empiezan por `E` con `filter`; encadenar `map`, `filter` y `reduce`; escribir `get_string_lists`, que deje solo las cadenas; sumar con `reduce` y formar con `reduce` la frase *Estonia, Finland, Sweden, Denmark, Norway, and Iceland are north European countries*.

In [5]:
from functools import reduce

countries = ["Estonia", "Finland", "Sweden", "Denmark", "Norway", "Iceland"]
names = ["Asabeneh", "Lidiya", "Ermias", "Abraham"]
numbers = list(range(1, 11))

print(list(map(str.upper, countries)), list(map(lambda n: n**2, numbers)), list(map(str.upper, names)))
print(list(filter(lambda c: "land" in c, countries)))
print(list(filter(lambda c: len(c) == 6, countries)), list(filter(lambda c: len(c) >= 6, countries)))
print(list(filter(lambda c: c.startswith("E"), countries)))
# suma de cuadrados de pares
print(reduce(lambda a, b: a + b, filter(lambda n: n % 2 == 0, map(lambda n: n**2, numbers))))

def get_string_lists(lista):
    return list(filter(lambda x: isinstance(x, str), lista))

print(get_string_lists([1, "a", 2.5, "b", None]))
print(reduce(lambda a, b: a + b, numbers))
frase = reduce(lambda a, b: f"{a}, {b}", countries[:-1]) + f", and {countries[-1]} are north European countries"
print(frase)

['ESTONIA', 'FINLAND', 'SWEDEN', 'DENMARK', 'NORWAY', 'ICELAND'] [1, 4, 9, 16, 25, 36, 49, 64, 81, 100] ['ASABENEH', 'LIDIYA', 'ERMIAS', 'ABRAHAM']
['Finland', 'Iceland']
['Sweden', 'Norway'] ['Estonia', 'Finland', 'Sweden', 'Denmark', 'Norway', 'Iceland']
['Estonia']
220
['a', 'b']
55
Estonia, Finland, Sweden, Denmark, Norway, and Iceland are north European countries


## 6. Funciones sobre la lista de países

Con la lista completa de países de *30 Days Of Python*: escribir `categorize_countries`, que devuelva los países que contienen un patrón (`land`, `ia`, `island`, `stan`); una función que devuelva un diccionario con cuántos países empiezan por cada letra; y `get_first_ten_countries` y `get_last_ten_countries`.

In [6]:
import ast
from collections import Counter
from urllib import request

URL_30_DIAS = "https://raw.githubusercontent.com/Asabeneh/30-Days-Of-Python/master/data/"
texto = request.urlopen(URL_30_DIAS + "countries.py").read().decode("utf-8")
todos = ast.literal_eval(texto[texto.index("["):].strip().rstrip(";"))

def categorize_countries(patron):
    return [c for c in todos if patron in c.lower()]

def paises_por_inicial():
    return dict(Counter(c[0] for c in todos))

def get_first_ten_countries():
    return todos[:10]

def get_last_ten_countries():
    return todos[-10:]

for patron in ("land", "ia", "island", "stan"):
    print(patron, len(categorize_countries(patron)), categorize_countries(patron)[:4])
print(paises_por_inicial())
print(get_first_ten_countries())
print(get_last_ten_countries())

land 10 ['Finland', 'Iceland', 'Ireland', 'Marshall Islands']
ia 40 ['Albania', 'Algeria', 'Armenia', 'Australia']
island 2 ['Marshall Islands', 'Solomon Islands']
stan 7 ['Afghanistan', 'Kazakhstan', 'Kyrgyzstan', 'Pakistan']
{'A': 11, 'B': 17, 'C': 18, 'D': 4, 'E': 9, 'F': 3, 'G': 11, 'H': 3, 'I': 8, 'J': 3, 'K': 7, 'L': 9, 'M': 18, 'N': 10, 'O': 1, 'P': 10, 'Q': 1, 'R': 3, 'S': 25, 'T': 10, 'U': 7, 'V': 4, 'Y': 1, 'Z': 2}
['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria']
['United States', 'Uruguay', 'Uzbekistan', 'Vanuatu', 'Vatican City', 'Venezuela', 'Vietnam', 'Yemen', 'Zambia', 'Zimbabwe']
